# Recovering message text from `attributedBody`

On recent macOS versions, `message.text` is often `NULL` and the body lives only in the
`attributedBody` blob (a serialized `NSAttributedString`, "typedstream" format). Filtering on
`text IS NOT NULL` silently drops those messages.

This notebook is read-only against `chat.db` and does not touch `testing.ipynb`.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path.home() / "Library/Messages/chat.db"


## 1. The decoder

The string follows the `NSString` class marker and 5 bytes of typedstream framing. It is prefixed
by a length: one byte if < 0x80, otherwise `0x81` + 2 bytes or `0x82` + 4 bytes (little-endian).

In [ ]:
def decode_attributed_body(blob: bytes | None) -> str | None:
    """Extract the plain text from a Messages `attributedBody` blob, or None if not found."""
    if blob is None:
        return None
    i = blob.find(b"NSString")
    if i < 0:
        return None
    rest = blob[i + len(b"NSString") + 5:]
    if not rest:
        return None
    n = rest[0]
    if n == 0x81:
        n, start = int.from_bytes(rest[1:3], "little"), 3
    elif n == 0x82:
        n, start = int.from_bytes(rest[1:5], "little"), 5
    else:
        start = 1
    return rest[start:start + n].decode("utf-8", errors="replace")


Sanity check on synthetic blobs (short, and long enough to need the 2-byte length):

In [ ]:
def fake_blob(text: str) -> bytes:
    body = text.encode()
    n = len(body)
    length = bytes([n]) if n < 0x80 else b"\x81" + n.to_bytes(2, "little")
    return (b"\x04\x0bstreamtyped\x81\xe8\x03\x84\x01@\x84\x84\x84\x12NSAttributedString"
            b"\x00\x84\x84\x08NSObject\x00\x85\x92\x84\x84\x84\x08NSString\x01\x94\x84\x01+"
            + length + body + b"\x86\x84\x02iI\x01")

for t in ["hi 🩷", "x" * 300]:
    assert decode_attributed_body(fake_blob(t)) == t
assert decode_attributed_body(None) is None
print("decoder OK")


## 2. Query without the `text IS NOT NULL` filter

Select `attributedBody` too, and keep the handle join so senders are available.

In [ ]:
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
df = pd.read_sql_query("""
    SELECT
        message.ROWID AS message_id,
        message.text,
        message.attributedBody,
        message.service,
        message.is_from_me,
        message.date,
        message.handle_id,
        handle.id AS sender
    FROM message
    LEFT JOIN handle ON message.handle_id = handle.ROWID
    WHERE message.is_from_me = 0
""", conn)
conn.close()
len(df)


## 3. How much did `text IS NOT NULL` hide?

In [ ]:
df["decoded"] = df["attributedBody"].map(decode_attributed_body)
df["text_full"] = df["text"].where(df["text"].notna(), df["decoded"])
recovered = df["text"].isna() & df["decoded"].fillna("").str.strip().ne("")

pd.Series({
    "incoming messages": len(df),
    "text column populated": int(df["text"].notna().sum()),
    "text NULL": int(df["text"].isna().sum()),
    "  ...recovered from attributedBody": int(recovered.sum()),
    "  ...still empty (attachments, reactions, etc.)": int((df["text"].isna() & ~recovered).sum()),
})


## 4. The messages from the screenshot (`handle_id == 1536`)

These had `text = NaN` in `testing.ipynb`.

In [ ]:
df.loc[df["handle_id"] == 1536, ["message_id", "service", "date", "text", "decoded"]]


## 5. Checking for decoding problems

Replacement characters (`\ufffd`) would indicate a decoding failure rather than a real message.

In [ ]:
bad = df["decoded"].fillna("").str.contains("\ufffd")
print("decoded messages containing U+FFFD:", int(bad.sum()))
df.loc[bad, ["message_id", "decoded"]].head()
